# Случайный лес HOMEWORK
дата сет с тойотами короллами. несмотря на то что задание с лин регрессий на гитхабе это задание дулаю с нуля

In [220]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from datetime import datetime
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.dummy import DummyRegressor
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import time

обзор данных был в ноутбуке с линейной регрессий. пропусков не было.


In [221]:
df = pd.read_csv("ToyotaCorolla.csv")

print("\n\nРазмеры:\n")
print(df.shape)



Размеры:

(1436, 39)


для лин регрессии цвет и тип топлив я кодировал. из поля с моделью в boosted версии доставал модель и тип кузова. в данном случае кодировать категориальные признаки тоже потребуется чтобы работала sickit learn реализация. но от кодирования цвета я наверное откажусь. Кроме того хоть и деревья более устойчивы к выбросам в признаках чем линейные модели все равно удалю их т.к. знаю какие.

In [222]:
df = df[df['CC'] <= 3000]
df = df[df['Weight'] <= 1600]
df = df[df['Gears'] > 4]

In [223]:
encoder = OneHotEncoder(sparse_output=False)

encoded_array = encoder.fit_transform(df[['Fuel_Type']])
new_columns = encoder.get_feature_names_out(['Fuel_Type'])
encoded_df = pd.DataFrame(encoded_array, columns=new_columns, index=df.index)
df = pd.concat([df, encoded_df], axis=1)

удаление части признаков

In [224]:
cols_to_drop = [
    'Id',
    'Model',
    'Mfg_Year',
    'Mfg_Month',
    'Cylinders',
    'Color',
    'Fuel_Type', # сделали кодирование
    'Radio_cassette',
    'Central_Lock'
]

df = df.drop(columns=[col for col in cols_to_drop if col in df.columns])

# РАЗБИЕНИЕ

In [225]:
y = df['Price']
X = df.drop('Price', axis=1)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, shuffle=True)

# ВОПРОСЫ #1:
1. Предобработку делал(смотреть выше). В отличие от линейной модели не делал масштабирования
2. 25% на тест и 75% на train с помощью train_test_split
3. Про кросс-валидацию: На 5 или 10 частей.
4. Учиться без кросс валидации можно. Это быстрее. В случае обучения без CV модель просто учится на данных выделенных под train.

# ОБУЧЕНИЕ СЛУЧАЙНОГО ЛЕСА И ДЕРЕВА

In [233]:
_depth = 15
start_time = time.time()

#моделька дерева решений
tree = DecisionTreeRegressor(max_depth=_depth, random_state=42)
tree.fit(X_train, y_train)

tree_train_time = time.time() - start_time
y_pred_tree = tree.predict(X_test)
tree_mae = mean_absolute_error(y_test, y_pred_tree)
tree_rmse = np.sqrt(mean_squared_error(y_test, y_pred_tree))
tree_r2 = r2_score(y_test, y_pred_tree)

# моделька случайного леса
start_time = time.time()
forest = RandomForestRegressor(n_estimators=100, max_depth = _depth, random_state = 42)
forest.fit(X_train, y_train)
forest_train_time = time.time() - start_time
y_pred_forest = forest.predict(X_test)
forest_mae = mean_absolute_error(y_test, y_pred_forest)
forest_rmse = np.sqrt(mean_squared_error(y_test, y_pred_forest))
forest_r2 = r2_score(y_test, y_pred_forest)
# модель случайного леса и обучение на всех ядрах cpu
start_time = time.time()
forest_multicore = RandomForestRegressor(n_estimators=100, max_depth = _depth, random_state = 42, n_jobs=-1)
forest_multicore.fit(X_train, y_train)
forest_multicore_train_time = time.time() - start_time

print("ДЕРЕВО РЕШЕНИЙ статистика")
print(f"Время обучения дерева(сек): {tree_train_time:.4f} сек")
print(f"MAE: {tree_mae:.2f}")
print(f"RMSE: {tree_rmse:.2f}")
print(f"R2: {tree_r2:.4f}")

print("\n\nСЛУЧАЙНЫЙ ЛЕС статистика")
print(f"Время обучения леса(сек): {forest_train_time:.4f} сек")
print(f"MAE: {forest_mae:.2f}")
print(f"RMSE: {forest_rmse:.2f}")
print(f"R2: {forest_r2:.4f}")


print("\n\nСЛУЧАЙНЫЙ ЛЕС результат на train:")
print(f"R2 на train: {r2_score(y_train, forest.predict(X_train)):.4f}")

print(f"\n\nВремя обучения на всех ядрах: {forest_multicore_train_time:.4f} сек")

ДЕРЕВО РЕШЕНИЙ статистика
Время обучения дерева(сек): 0.0163 сек
MAE: 1049.89
RMSE: 1423.29
R2: 0.8343


СЛУЧАЙНЫЙ ЛЕС статистика
Время обучения леса(сек): 0.7762 сек
MAE: 738.88
RMSE: 979.13
R2: 0.9216


СЛУЧАЙНЫЙ ЛЕС результат на train:
R2 на train: 0.9871


Время обучения на всех ядрах: 0.6297 сек


# ВОПРОСЫ #2:
1. Простое дерево обучается кртано быстрее. Это так ибо лес в данном случае состоит из 100 деревьев
2. Принципиально на одном ядре ничего не сделать, если нельзя менять параметры леса(количество деревьев и глубина). На лекции озвучивался варинт обучения на нескольких ядрах. Я попробовал n_jobs=-1. при обучении на всех 4х ядрах время сократилось примерно на 20%. но даже эта дельта все равно кратно больше времени обучения одного дерева.
3. Случайный лес значительно лучше. Снижение дисперсии при сохранении bias делает своё дело.

# СЛУЧАЙНЫЙ ЛЕС с подбором гиперпараметров и кросс валидацией.
попробую еще улучшить результат предсказания



In [227]:
param_grid = {
    'n_estimators': [10, 50, 100],
    'max_depth': [7,10, 15, 20]
}

forest = RandomForestRegressor(random_state = 42, n_jobs = -1)
grid_search = GridSearchCV(estimator=forest, param_grid=param_grid, cv=5, scoring='r2', n_jobs=-1, verbose=1)

grid_search.fit(X_train, y_train)
forest_ultra = grid_search.best_estimator_

y_pred_forest_ultra = forest_ultra.predict(X_test)

forest_ultra_mae = mean_absolute_error(y_test, y_pred_forest_ultra)
forest_ultra_rmse = np.sqrt(mean_squared_error(y_test, y_pred_forest_ultra))
forest_ultra_r2 = r2_score(y_test, y_pred_forest_ultra)



print("\n\nСЛУЧАЙНЫЙ ЛЕС forest_ultra статистика")
print(f"MAE: {forest_ultra_mae:.2f}")
print(f"RMSE: {forest_ultra_rmse:.2f}")
print(f"R2: {forest_ultra_r2:.4f}")

Fitting 5 folds for each of 12 candidates, totalling 60 fits


СЛУЧАЙНЫЙ ЛЕС forest_ultra статистика
MAE: 744.98
RMSE: 983.98
R2: 0.9208


In [228]:
print(grid_search.best_params_)

{'max_depth': 10, 'n_estimators': 100}


гиперпараметры отличаются от тех что использовал выше... как будто по приколу можно два леса еще раз обучить с кросс валидацией и посмотреть какой более стабильный. с глубиной 10 или 15

In [229]:
cv = KFold(n_splits=5, shuffle=True, random_state=42)
forest_depth10 = RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1)
forest_depth15 = RandomForestRegressor(n_estimators=100, max_depth=15, random_state=42, n_jobs=-1)
scores_depth10 = cross_val_score(forest_depth10, X_train, y_train, cv=cv, scoring='r2', n_jobs=-1)
scores_depth15 = cross_val_score(forest_depth15, X_train, y_train, cv=cv, scoring='r2', n_jobs=-1)

print(f"Среднее по кросс-валидации глубина 10: {(scores_depth10.mean()):.4f}")
print(f"Среднее по кросс-валидации глубина 15: {(scores_depth15.mean()):.4f}")

Среднее по кросс-валидации глубина 10: 0.8979
Среднее по кросс-валидации глубина 15: 0.8969


в целом моделька с максимльной глубиной 10 самую малость но стабильнее. а потому потому в gridsearch такие лучшие параметры. более высокий результат на тесте где глубина 15 спишем на удачное разбиение датасета.(можно сид поменять) интересный эксперементик

# И ФИНАЛЬНОЕ(ВОПРОСЫ #3):
1. Использовал несколько. RMSE т.к. она более чувтсвительна к бОльшим отклонениям. MAE т.к. весьма просто интерпретируется(самое то для простого обывателя чтобы знать сколько лишних денег везти с собой). и R2 - метрика которая легко интерпертируется в процентах. 0.92 обозначает что модель объясняет 92% изменчивасти цены. Универсальная метрика т.к. может служить для сравнения качества разных моделей.
2. На тесте.
3. Лучше справился случайный лес. Линейная регрессия в первой своей вариации у меня выдала r2= 0.904 при идентичном разбиении. С тем инжинирингом фичей который мне удалось реализовать модель показала 0.919 (смотреть ноутбук который подписан BOOSTED). Здесь же случайный лес с простым подбором параметров через gridsearch показал такой же результат как и линейная регрессия после апгрейда.
4. Мне думается достаточно хорошие.